[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/lekevin01/AI_Image_Detection_01/blob/main/CECS456_Project1_AI_Image_Detection.ipynb)

*Run it in Colab — the first code cell downloads CIFAKE, nothing to upload.*


# CECS 456 — Project 1: AI-Generated Image Detection

**Task:** binary classification — is a 32×32 image **FAKE** (AI-generated) or **REAL** (photograph)?  
**Dataset:** CIFAKE (Bird & Lotfi, 2024) — 100,000 train / 20,000 test, 50/50 FAKE/REAL  
**Framework:** PyTorch + torchvision

![The forward pass: image -> tensor -> model -> logit -> P(REAL)](https://raw.githubusercontent.com/lekevin01/AI_Image_Detection_01/main/diagrams/pipeline.svg)

## Contents

| Part | Topic |
|---|---|
| 0 | Setup — seeds, device |
| 1 | Load & inspect the data |
| 2 | Images → tensors → mini-batches |
| 3 | Reproducible train / val / test split |
| 4 | Documentation — preprocessing + citations |
| 5 | The ideas — neuron → layer → loss → training |
| 6 | MLP baseline |
| 7 | Custom CNN |
| 8 | Train the CNN — run_epoch, Adam, learning curves |
| 9 | Metrics — F1, ROC-AUC, confusion matrix |
| 10 | Robustness — JPEG / blur / noise |
| 11 | Fine-tuned pretrained ResNet |
| 12 | Error analysis + summary + citations |

Dataset source: https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images

## Learning objectives (data stage)

By the end of this notebook you should be able to:

1. Load CIFAKE with `ImageFolder`, explain the label mapping, and check class balance.
2. Explain what a **tensor** is, why images become `[N, C, H, W]` tensors, and why we normalize pixels.
3. Explain what a **mini-batch** and an **epoch** are, and do the batch arithmetic.
4. Create a **reproducible** train / validation / test split and explain the job of each set.
5. **Document** every step: official docs links, preprocessing choices explained, and citations (a Project 1 requirement).

## How to work through this notebook

- Run every cell **top to bottom**.
- Read the note **before** each code cell, and answer the **Question:** boxes in your own words *before* running it. Tell me your answers in chat — we check them together.
- If a cell surprises you, stop and ask. "The code ran" is not the goal; *understanding why* is.

## How the Project 1 requirements map to the plan

| Requirement (from the assignment PDF) | Where we handle it |
|---|---|
| Separate training, validation, and test data | this notebook, Part 3 |
| Explain preprocessing and experimental settings | this notebook, Parts 2 & 4 |
| Train a CNN-based classifier (custom CNN and/or fine-tuned ResNet) | later stages, together |
| Evaluate on clean **and modified** test images (≥2 of: JPEG, blur, noise) | robustness stage |
| Report F1, ROC-AUC, confusion matrix | evaluation stage |
| Show representative correct and incorrect predictions | error-analysis stage |
| Discuss limits of the detector on other image sources | report stage |
| Cite datasets, pretrained models, and external code | this notebook (Part 4) and the report |

## The whole picture

How the pieces relate to **your** project (CIFAKE: is this 32×32 image FAKE or REAL?):

![CIFAKE training loop](https://raw.githubusercontent.com/lekevin01/AI_Image_Detection_01/main/diagrams/project_loop.svg)

**Read the loop:** image → tensor → model → logit → sigmoid → P(REAL) → compare with the true label → loss → `backward()` computes gradients → `step()` nudges the weights → repeat. That loop *is* training.

![One neuron, a layer, your network](https://raw.githubusercontent.com/lekevin01/AI_Image_Detection_01/main/diagrams/neuron_to_network.svg)

- **One neuron** = weighted sum + bias + activation → one line.
- **A layer** = many neurons in parallel → many lines.
- **Your MLP** = 3,072 inputs → 128 hidden → 1 output. The hidden neurons learn 128 "ingredients"; the last one combines them into FAKE vs REAL.

> The videos teach with coffee and exam scores. These diagrams map the same ideas onto your images. If an image does not show, open `https://raw.githubusercontent.com/lekevin01/AI_Image_Detection_01/main/diagrams/project_loop.svg` in a browser.

## The project folder

```text
PROJECT/
├── archive/
│   ├── train/               100,000 images
│   │   ├── FAKE/
│   │   └── REAL/
│   └── test/                20,000 images
│       ├── FAKE/
│       └── REAL/
└── CECS456_Project1_AI_Image_Detection.ipynb   ← this notebook
```

Every image is **32×32 RGB**, and the label is the name of the folder it sits in. This notebook must stay in the project folder that contains `archive/`.

---
# Part 0. Setup

We use **PyTorch**, not TensorFlow. Two ideas in this cell:

- `torch.manual_seed(0)` fixes the random numbers, so every run starts with the **same random values** — part of "explain your experimental settings", a general project requirement.
- `device` is a name for *where* the math runs: `cuda` (NVIDIA GPU), `mps` (Apple Silicon), or `cpu`. The code picks a GPU if one exists and falls back to the CPU.

Documentation: [`torch`](https://pytorch.org/docs/stable/index.html) · [`torch.manual_seed`](https://pytorch.org/docs/stable/generated/torch.manual_seed.html) · [`torch.cuda.is_available`](https://pytorch.org/docs/stable/generated/torch.cuda.is_available.html)

In [ ]:
import sys
from pathlib import Path
from collections import Counter

import torch
import torch.nn as nn
from torch import optim
from torch.utils.data import DataLoader, Subset, random_split
import torchvision
from torchvision import datasets
from torchvision.transforms import v2
import matplotlib.pyplot as plt

torch.manual_seed(0)   # same random values every run

# Where the math runs: cuda (NVIDIA GPU), mps (Apple Silicon), or cpu
if torch.cuda.is_available():
    device = "cuda"
elif getattr(torch.backends, "mps", None) is not None and torch.backends.mps.is_available():
    device = "mps"
else:
    device = "cpu"

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("torchvision:", torchvision.__version__)
print("Using device:", device)

A CPU-only installation prints `Using device: cpu` — expected, and enough for the whole data stage.

## Getting the data

Every cell below reads from `archive/`. That folder is **downloaded from a public
link**, so this notebook runs for anyone — no Kaggle account, no API key, no
Drive mount, no upload:

    https://drive.google.com/file/d/1hsJTZwrmPzQW0KoiyLmf_M-TjS71HB7_/view

About 110 MiB and 120,000 small files, once per runtime. If `archive/` is already
next to the notebook (your own machine), the cell skips the download.


In [ ]:
# Get the dataset from a public link: no account, no key, no Drive mount.
# Anyone who opens this notebook can run this cell and get the same files.
import subprocess
import sys
from pathlib import Path

data_root = Path("archive")
FILE_ID = "1hsJTZwrmPzQW0KoiyLmf_M-TjS71HB7_"
ZIP = Path("/content/archive.zip")

if data_root.exists():
    print("Using the existing folder:", data_root.resolve())
else:
    print("Downloading the dataset (~110 MiB) from the public Drive link ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
    got = subprocess.run([sys.executable, "-m", "gdown", "--id", FILE_ID, "-O", str(ZIP)])

    if got.returncode != 0 or not ZIP.exists():
        # Fallback: plain HTTP, with the large-file confirmation flag set.
        subprocess.run(
            ["curl", "-L",
             "https://drive.google.com/uc?export=download&id=" + FILE_ID + "&confirm=t",
             "-o", str(ZIP)],
            check=True,
        )

    size = ZIP.stat().st_size if ZIP.exists() else 0
    if size < 50_000_000:
        raise SystemExit(
            "Download failed or was blocked (" + f"{size:,} bytes). "
            "Open the Drive link, confirm it is shared as 'Anyone with the link', "
            "then re-run this cell."
        )

    subprocess.run(["unzip", "-q", "-o", str(ZIP), "-d", "/content"], check=True)

    if not data_root.exists():
        # Some archives keep train/ and test/ at the root instead of archive/.
        if Path("/content/train").is_dir() and Path("/content/test").is_dir():
            data_root.mkdir()
            for part in ("train", "test"):
                Path("/content", part).rename(data_root / part)

    if not data_root.exists():
        raise SystemExit(
            "Unpacked, but found neither archive/ nor train/ + test/. "
            "Check what the zip contains and report it."
        )

    print("Unpacked ->", data_root.resolve())

print("Contents:", sorted(p.name for p in data_root.iterdir()))


---
# Part 1. Load and inspect the data

`Path` represents a filesystem path. `Path("archive")` is a **relative** path.

**Question:** relative to what? (Hint: where does Jupyter run code from?)

Documentation: [`pathlib.Path`](https://docs.python.org/3/library/pathlib.html#pathlib.Path)

In [ ]:
data_root = Path("archive")

print("Dataset folder exists:", data_root.exists())
print("Training folder exists:", (data_root / "train").exists())
print("Test folder exists:", (data_root / "test").exists())

All three results must be `True`. The cell above downloads the dataset into `archive/` when it is missing, so a failure here now means the download did not finish rather than a wrong working directory. Check with `Path.cwd()`.


### Labels: folder names become numbers

[`ImageFolder`](https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html) reads images from class-named folders and assigns an integer label per folder, in **alphabetical order**.

**Question:** `FAKE` and `REAL` — which one gets label 0 and which gets label 1? Predict, then run the cell.

In [ ]:
# No transform yet: items are (PIL image, integer label)
train_dataset = datasets.ImageFolder(root=data_root / "train")
test_dataset = datasets.ImageFolder(root=data_root / "test")

print("Training images:", len(train_dataset))
print("Test images:", len(test_dataset))
print("Class-to-label mapping:", train_dataset.class_to_idx)
print("Classes:", train_dataset.classes)

Expected:

```text
Training images: 100000
Test images: 20000
Class-to-label mapping: {'FAKE': 0, 'REAL': 1}
```

So from now on: **0 = FAKE, 1 = REAL** (F sorts before R).

### Class balance

If one class dominates, a lazy model can score high accuracy by always predicting the majority class — without learning anything. Checking balance **before** training catches this trap.

**Question:** CIFAKE has 100,000 train and 20,000 test images. Is it balanced? Predict the FAKE/REAL percentages, then run the cell.

`ImageFolder.targets` holds the label of every image without loading any pixels, so this loop is fast.

In [ ]:
fake_idx = train_dataset.class_to_idx["FAKE"]
real_idx = train_dataset.class_to_idx["REAL"]

for name, ds in [("train", train_dataset), ("test", test_dataset)]:
    counts = Counter(ds.targets)
    total = len(ds)
    print(f"{name}: {total} images | FAKE {counts[fake_idx]} ({counts[fake_idx] / total:.1%}) | REAL {counts[real_idx]} ({counts[real_idx] / total:.1%})")

### Look at the data

Always inspect examples before training. This can reveal wrong paths, wrong labels, corrupt files, or broken transforms. At 32×32, deciding FAKE vs REAL is genuinely hard — good, that is the whole point of the project.

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(8, 4))

for i, ax in enumerate(axes.flat):
    image, label = train_dataset[i]          # PIL image + integer label
    ax.imshow(image)
    ax.set_title(train_dataset.classes[label])
    ax.axis("off")

plt.tight_layout()
plt.show()

**Checkpoint 1.** You have verified the dataset location, confirmed the label mapping (`0 = FAKE`, `1 = REAL`), checked that both classes are balanced, and looked at real examples.

---
# Part 2. Images → tensors → mini-batches

A neural network does not see JPG files. It sees **tensors**: containers of numbers.

- One image becomes a tensor of shape **[C, H, W]** = channels, height, width → `[3, 32, 32]` for RGB 32×32.
- One batch stacks N images into **[N, C, H, W]**.
- Pixel bytes (0–255) are scaled to floats (0–1) so the math behaves.

📺 [StatQuest playlist](https://www.youtube.com/playlist?list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1) — video #26 "Tensors for Neural Networks".

### The transform, step by step

| Step | What it does | Range after |
|---|---|---|
| `v2.ToImage()` | PIL → tensor | still 0–255 |
| `v2.ToDtype(torch.float32, scale=True)` | integers → floats, divide by 255 | 0.0–1.0 |
| `v2.Normalize((0.5,)*3, (0.5,)*3)` | `(x − 0.5) / 0.5` per channel | −1.0–1.0 |

Normalizing centers the inputs around 0 — remember the coffee lab: large inputs push activations into flat regions where gradients almost vanish, so training barely moves. (We will revisit this when we build the model; for now: this is a **preprocessing choice**, and you must be able to explain it.)

**Question:** after this transform, what will the **minimum and maximum pixel value** of a batch be?

Documentation: [`torchvision.transforms.v2`](https://pytorch.org/vision/stable/transforms.html) · [`v2.Compose`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.Compose.html) · [`v2.Normalize`](https://pytorch.org/vision/stable/generated/torchvision.transforms.v2.Normalize.html)

In [ ]:
transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),      # 0-255 -> 0.0-1.0
    v2.Normalize(mean=(0.5, 0.5, 0.5),
                 std=(0.5, 0.5, 0.5)),          # 0.0-1.0 -> -1.0-1.0
])

train_dataset = datasets.ImageFolder(root=data_root / "train", transform=transform)
test_dataset = datasets.ImageFolder(root=data_root / "test", transform=transform)

print("Transform ready:", transform)

### Mini-batches with `DataLoader`

We never feed all 100,000 images at once. A [`DataLoader`](https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader) serves them in **mini-batches** and can shuffle them each epoch. One **epoch** = every training example goes through the model once. Each batch gets **one** weight update, so:

```text
batches per epoch = training images / batch_size
                   = 100,000 / 64 = 1562.5  ->  1563 batches (the last one is smaller)
```

**Question:** we set `shuffle=True` for training and `shuffle=False` for the test set. Why does the test set not need shuffling?

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=512, shuffle=False)

print("Batches per epoch (full train set):", len(train_loader))
print("Test batches:", len(test_loader))

### Mini-batch vs. full batch vs. SGD

How many examples do we use for each update?

| Name | Examples per update | Updates per epoch (100,000 images) |
|---|---|---|
| Batch gradient descent | all 100,000 | 1 |
| SGD (textbook version) | 1 | 100,000 |
| Mini-batch SGD | 64 | ≈ 1563 |

- Mini-batch is the middle ground: faster than full batch, less noisy than batch size 1.
- In practice, when people say "SGD", they almost always mean **mini-batch SGD**.
- The batch size lives in the `DataLoader`, **not** in the optimizer.

### Inspect one batch

`next(iter(train_loader))` means "give me the first batch". A `DataLoader` is normally used in a `for` loop; this is just a quick peek.

**Question:** predict the four values this cell prints — image shape, label shape, dtype, pixel range.

In [ ]:
images, labels = next(iter(train_loader))

print("Image batch shape:", images.shape)   # [N, C, H, W]
print("Label batch shape:", labels.shape)
print("Image data type:", images.dtype)
print("Pixel range:", images.min().item(), "to", images.max().item())
print("First 8 labels:", labels[:8].tolist())
print("Classes:", train_dataset.classes)

Expected:

```text
Image batch shape: torch.Size([64, 3, 32, 32])
Label batch shape: torch.Size([64])
Image data type: torch.float32
Pixel range: -1.0 to 1.0
```

**Checkpoint 2.** Images are float tensors in [−1, 1], served 64 at a time. Labels are plain integers (0/1), one per image.

---
# Part 3. A reproducible train / validation / test split

The assignment requires **separate training, validation, and test data**. Three sets, three jobs:

- **Train** (90,000): the model learns from it.
- **Validation** (10,000): we watch progress during training and tune hyperparameters on it.
- **Test** (20,000): used **only for evaluation, never for training** — metrics, robustness tests, and error analysis all happen on this set later.

`random_split` with a **seeded generator** makes the split reproducible: every student, every run, gets the same 90k/10k cut. That is also part of "explain your experimental settings".

Another requirement to remember: **related images and augmented copies must stay within the same split.** Our split obeys this automatically — and later, every modified image (JPEG/blur/noise) will be generated **only from test images**, so no modified copy ever leaks into training.

Documentation: [`random_split`](https://pytorch.org/docs/stable/data.html#torch.utils.data.random_split) · [`torch.Generator`](https://pytorch.org/docs/stable/generated/torch.Generator.html)

In [ ]:
train_set, val_set = random_split(train_dataset, [90000, 10000],
                                  generator=torch.Generator().manual_seed(0))

print("train:", len(train_set), " val:", len(val_set), " test:", len(test_dataset))

### Verify the balance survived the split

A `Subset` stores its chosen positions in `.indices`, so we can count classes without loading a single image.

**Question:** why do we care whether the split is balanced, if the full dataset is balanced?

In [ ]:
def balance_of(subset, source):
    """Count FAKE/REAL inside a split, using its stored indices (no image loading)."""
    fake_idx = source.class_to_idx["FAKE"]
    real_idx = source.class_to_idx["REAL"]
    targets = [source.targets[i] for i in subset.indices]
    return targets.count(fake_idx), targets.count(real_idx)

for name, subset in [("train", train_set), ("val", val_set)]:
    fake, real = balance_of(subset, train_dataset)
    total = len(subset)
    print(f"{name}: {total} images | FAKE {fake} ({fake / total:.1%}) | REAL {real} ({real / total:.1%})")

### CPU-friendly training subset + the three loaders

Training on all 90,000 images takes a while on a laptop CPU. So we take a **reproducible 5,000-image slice** of the training set by default. Flip `SUBSET = None` when you move to Colab with a GPU.

**Question:** with 5,000 images and batch size 64, how many weight updates happen per epoch?

In [ ]:
SUBSET = 5000   # images per epoch on a laptop CPU; set to None for all 90,000 (Colab GPU)

if SUBSET is not None:
    idx = torch.randperm(len(train_set), generator=torch.Generator().manual_seed(0))[:SUBSET]
    train_sub = Subset(train_set, idx)
else:
    train_sub = train_set

train_loader = DataLoader(train_sub, batch_size=64, shuffle=True)
val_loader = DataLoader(val_set, batch_size=512, shuffle=False)

print("Training images this run:", len(train_sub))
print("Batches per epoch (train):", len(train_loader))
print("Batches in val set:", len(val_loader))

**Checkpoint 3.** You have a reproducible 90k/10k split, both classes stay balanced in every split, and the three loaders are ready. The test set has only been used for a peek — never for training.

---
# Part 4. Documentation (a Project 1 requirement)

The assignment asks you to **explain preprocessing, model selection, evaluation metrics, and experimental settings**, and to **cite all datasets, pretrained models, and external code**. This section is the data-stage half of that requirement — fill it in as you go, and it becomes part of your final report.

### 4.1 Preprocessing choices (explain these in your own words)

| Choice | What we did | Why |
|---|---|---|
| Image format | 32×32 RGB, PIL → tensor | networks consume tensors, not files |
| Scaling | pixels 0–255 → floats 0–1 | keeps the math in a small, predictable range |
| Normalization | `(x − 0.5) / 0.5` per channel → [−1, 1] | centers inputs around 0, which helps training converge |
| Batching | 64 per training batch, 512 for evaluation | mini-batch SGD: faster than full batch, less noisy than batch size 1 |
| Shuffling | train yes, val/test no | the model must not learn the order; evaluation order doesn't matter |
| Splitting | seeded 90k train / 10k val / 20k test | separate sets per requirement; seed makes it reproducible |
| Subsetting | seeded 5,000-image train slice on CPU | full training is for Colab GPU; slice keeps CPU runs fast |

### 4.2 Documentation links used so far

| API | Official docs |
|---|---|
| `pathlib.Path` | https://docs.python.org/3/library/pathlib.html |
| `datasets.ImageFolder` | https://pytorch.org/vision/stable/generated/torchvision.datasets.ImageFolder.html |
| `torchvision.transforms.v2` | https://pytorch.org/vision/stable/transforms.html |
| `DataLoader` | https://pytorch.org/docs/stable/data.html#torch.utils.data.DataLoader |
| `random_split` | https://pytorch.org/docs/stable/data.html#torch.utils.data.random_split |
| `torch.manual_seed` | https://pytorch.org/docs/stable/generated/torch.manual_seed.html |

### 4.3 Citations

- **Dataset:** CIFAKE — Real and AI-Generated Synthetic Images (Bird & Lotfi, 2024), https://www.kaggle.com/datasets/birdy654/cifake-real-and-ai-generated-synthetic-images
- **Framework:** PyTorch, https://pytorch.org/ · torchvision, https://pytorch.org/vision/
- **Learning resource:** StatQuest Neural Networks playlist, https://www.youtube.com/playlist?list=PLblh5JKOoLUIxGDQs4LFFD--41Vzf-ME1

> Add citations here as you use more resources (pretrained models, scikit-learn, etc.) — the assignment requires citing every external piece.

---
# Part 5. The ideas: neuron → layer → loss → training

No code here — the four ideas every neural network is built from.

### 5.1 One neuron

```text
x1 ──w1──┐
x2 ──w2──┼── z = w1·x1 + w2·x2 + b ── activation ──> a
x3 ──w3──┘
```

- **weights** and **bias** are the parameters training adjusts.
- The **activation** makes the output non-linear.
- One neuron with a sigmoid = **logistic regression**: a single straight-line boundary.

**Question:** one neuron = one line. Where is one line not enough?

📺 playlist **#02** The Essential Main Ideas of Neural Networks · **#09** Multiple Inputs and Outputs

### 5.2 Layers, hidden units, and ReLU

- A **layer** is many neurons in parallel: `nn.Linear(D, E)` takes D inputs, produces E outputs.
- A **hidden layer** combines several lines into bendy boundaries — the two-step "ridge" the coffee lab needed.
- **ReLU** = `max(0, x)`: the default hidden-layer activation. It does not saturate for large inputs, so gradients stay healthy.
- Two `Linear` layers with **no** activation collapse into one `Linear` — the network can only draw a straight line again.

**Question:** two Linear layers with no activation between them behave like what?

📺 playlist **#08** ReLU In Action

### 5.3 The loss

> **The loss is one number that says how wrong the model is. Training means making it smaller.**

Binary classification uses **binary cross-entropy** (`BCEWithLogitsLoss`). A model guessing 50/50 has loss `−ln(0.5) ≈ 0.69`.

**Question:** your loss stays at 0.69 after training. What does that tell you?

📺 playlist **#12** Cross Entropy

### 5.4 Gradient descent + backpropagation = the five-step loop

| Code | Plain English |
|---|---|
| `optimizer.zero_grad()` | erase old gradients |
| `pred = model(X)` | guess (forward pass) |
| `loss = loss_fn(pred, y)` | measure how wrong |
| `loss.backward()` | figure out the gradients (backpropagation) |
| `optimizer.step()` | nudge the parameters (gradient descent) |

- **Gradient descent:** `w ← w − lr × gradient`. The learning rate `lr` is the step size.
- **Backpropagation:** the chain rule applied backwards through the network.
- `backward()` computes gradients; only `step()` changes weights.

📺 playlist **#04** Gradient Descent, Step-by-Step · **#05** Backpropagation Main Ideas

### Checkpoint 5 — explain it back

1. What does a single neuron compute?
2. Why do we put a ReLU between two `Linear` layers?
3. What does a loss of 0.69 mean?
4. Put these in order: `step`, `model(X)`, `backward`, `zero_grad`, `loss_fn`.

Click each box to check yourself:

<details><summary>Answer 1</summary>A weighted sum of its inputs plus a bias, passed through an activation: `a = f(w·x + b)`.</details>
<details><summary>Answer 2</summary>ReLU adds non-linearity. Without it, two `Linear` layers collapse into one, so the network could only draw a straight line.</details>
<details><summary>Answer 3</summary>0.69 ≈ −ln(0.5): the model is guessing 50/50 — it has not learned anything yet.</details>
<details><summary>Answer 4</summary>`zero_grad` → `model(X)` → `loss_fn` → `backward` → `step`.</details>
<details><summary>5.1's question</summary>When good results sit between two bad regions (coffee: good between too-cold and too-hot) — one line cannot carve a middle band; two steps (a hidden layer) can.</details>

---
# Part 6. The first model: an MLP on raw pixels

The simplest network that can work. One batch flows through:

```text
[B, 3, 32, 32] --flatten--> [B, 3072] --Linear(3072, 128)--> ReLU --> Linear(128, 1) --> [B, 1]
```

- **Flatten** turns each 32×32×3 image into 3,072 numbers.

![Flatten: 3 grids become one row](https://raw.githubusercontent.com/lekevin01/AI_Image_Detection_01/main/diagrams/flatten.svg)
- The output is one raw score per image — a **logit**. There is **no sigmoid in the model**: `BCEWithLogitsLoss` applies it internally.
- To display probabilities later: `torch.sigmoid(logits)`; decision at 0.5.

Documentation: [`nn.Module`](https://pytorch.org/docs/stable/generated/torch.nn.Module.html) · [`nn.Linear`](https://pytorch.org/docs/stable/generated/torch.nn.Linear.html) · [`nn.Flatten`](https://pytorch.org/docs/stable/generated/torch.nn.Flatten.html) · [`nn.ReLU`](https://pytorch.org/docs/stable/generated/torch.nn.ReLU.html)

In [ ]:
class MLP(nn.Module):
    def __init__(self, in_features = 3 * 32 * 32, hidden = 128):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(in_features, hidden)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden, 1)


    def forward(self, x):
        x = self.flatten(x)
        x = self.relu(self.fc1(x))
        return self.fc2(x)


model = MLP().to(device)
print(model)
print("Parameters: ", sum(p.numel() for p in model.parameters() ))

**Checkpoint 6.** The print-out should show **393,473** parameters: `fc1` = 3072×128 + 128 = 393,344; `fc2` = 128 + 1 = 129. Run it, paste the output, and we'll train it next.

---
# Part 7. The custom CNN

An MLP treats a 32×32 image as 3,072 unrelated numbers. But nearby pixels belong together, and the same pattern means the same thing wherever it appears. A **convolutional** network uses that:

- **3×3 filter:** slides across the image; one filter = one pattern detector (edge, blob, texture).
- **ReLU:** keeps strong responses.
- **Max pool 2×2:** keeps the strongest value per block — the map shrinks 32→16→8.
- **Flatten → Linear:** the classifier head.

![CNN on a 32×32 image](https://raw.githubusercontent.com/lekevin01/AI_Image_Detection_01/main/diagrams/cnn.svg)

Architecture: `Conv2d(3,32,3,padding=1)` → ReLU → pool → `Conv2d(32,64,3,padding=1)` → ReLU → pool → flatten → `Linear(4096,128)` → ReLU → `Linear(128,1)`.

**Question:** how many parameters? Count by hand, then check the print-out:

```text
conv1: 3×32×(3×3) + 32 = 896
conv2: 32×64×(3×3) + 64 = 18,496
fc1:   4096×128 + 128   = 524,416
fc2:   128 + 1          = 129
total:                   = 543,937
```

Documentation: [`nn.Conv2d`](https://pytorch.org/docs/stable/generated/torch.nn.Conv2d.html) · [`nn.MaxPool2d`](https://pytorch.org/docs/stable/generated/torch.nn.MaxPool2d.html)

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super().__init__()

        # --- Block 1: extract simple patterns ---
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)  # 3 RGB -> 32 feature maps
        self.pool  = nn.MaxPool2d(kernel_size=2, stride=2)       # halves size: 32 -> 16

        # --- Block 2: combine into bigger patterns ---
        self.conv2 = nn.Conv2d(32, 64, kernel_size=3, padding=1) # 32 -> 64 feature maps

        # --- Classifier head (same idea as the MLP) ---
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64 * 8 * 8, 128)                    # 4096 -> 128
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(128, 1)                             # 128 -> 1 logit

    def forward(self, x):
        x = self.relu(self.conv1(x))   # [B, 3, 32, 32] -> [B, 32, 32, 32]
        x = self.pool(x)               #                -> [B, 32, 16, 16]
        x = self.relu(self.conv2(x))   #                -> [B, 64, 16, 16]
        x = self.pool(x)               #                -> [B, 64,  8,  8]
        x = self.flatten(x)            #                -> [B, 4096]
        x = self.relu(self.fc1(x))     #                -> [B, 128]
        return self.fc2(x)             #                -> [B, 1]   (logits, NO sigmoid)

model = CNN().to(device)
print(model)
print("Parameters:", sum(p.numel() for p in model.parameters()))

**Checkpoint 7.** The print-out should show **543,937** parameters. Paste it, then Part 8 trains this model.

---
# Part 8. Train the CNN (following `demo_cnn_pytorch.ipynb`)

Same recipe as the lab demo, adapted to 2 classes: one `run_epoch(model, loader, optimizer=None)` function that **trains** when an optimizer is passed and **evaluates** when it is not, returning loss and accuracy. Then 5 epochs, plot learning curves, evaluate on test, and predict a few test images.

**Binary adaptations:** `nn.BCEWithLogitsLoss` instead of `CrossEntropyLoss`, and `sigmoid(...) >= 0.5` instead of `argmax`.

**Adam** = adaptive gradient descent: `w <- w - lr * m_hat / (sqrt(v_hat) + eps)` — the course's `lr` is kept, the raw gradient is smoothed (`m_hat`) and per-weight normalized (`sqrt(v_hat)`).

**Read before running:** untrained loss starts near `0.69` and accuracy near `0.50`; after training, loss falls and accuracy climbs.

In [ ]:
criterion = nn.BCEWithLogitsLoss()            # sigmoid + binary cross-entropy (demo uses CrossEntropyLoss for 10 classes)
optimizer = torch.optim.Adam(model.parameters())   # Adam with its default lr=1e-3, like the demo

In [ ]:
def run_epoch(model, loader, optimizer=None):
    # One pass over a DataLoader. Pass an optimizer to train; omit it to evaluate.
    train = optimizer is not None
    model.train() if train else model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0

    context = torch.enable_grad() if train else torch.no_grad()
    with context:
        for images, labels in loader:
            images = images.to(device)
            labels = labels.to(device)

            logits = model(images)                                # [B, 1]
            loss = criterion(logits.squeeze(1), labels.float())   # BCE wants [B] logits + float labels

            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            total_loss += loss.item() * labels.size(0)
            preds = (torch.sigmoid(logits.squeeze(1)) >= 0.5).long()   # 0=FAKE, 1=REAL
            total_correct += (preds == labels).sum().item()
            total += labels.size(0)

    return total_loss / total, total_correct / total

In [ ]:
EPOCHS = 5
history = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}

for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(model, train_loader, optimizer)
    val_loss, val_acc = run_epoch(model, val_loader)

    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)

    print(
        f'Epoch {epoch}/{EPOCHS}  '
        f'train loss {train_loss:.4f}  acc {train_acc:.4f}  '
        f'val loss {val_loss:.4f}  acc {val_acc:.4f}'
    )

In [ ]:
epochs = range(1, EPOCHS + 1)

plt.figure(figsize=(10, 4))

plt.subplot(1, 2, 1)
plt.plot(epochs, history['train_loss'], marker='o', label='train')
plt.plot(epochs, history['val_loss'], marker='o', label='val')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Loss')
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(epochs, history['train_acc'], marker='o', label='train')
plt.plot(epochs, history['val_acc'], marker='o', label='val')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Accuracy')
plt.legend()

plt.tight_layout()
plt.show()

In [ ]:
test_loss, test_acc = run_epoch(model, test_loader)
print('Total loss on Testing Set:', f'{test_loss:.4f}')
print('Accuracy of Testing Set:', f'{test_acc:.4f}')

In [ ]:
model.eval()
X_new = torch.stack([test_dataset[i][0] for i in range(3)])   # [3, 3, 32, 32]
y_new = torch.tensor([test_dataset[i][1] for i in range(3)])  # true labels 0/1

with torch.no_grad():
    logits = model(X_new.to(device))
    p_real = torch.sigmoid(logits).squeeze(1).cpu()            # P(REAL) for each

y_pred = (p_real >= 0.5).long()
classes = test_dataset.classes   # ['FAKE', 'REAL']
print('P(REAL) rounded:', [round(float(p), 2) for p in p_real])
print('Predicted:', [classes[i] for i in y_pred])
print('True:     ', [classes[i] for i in y_new])

---
# Part 9. Metrics: F1, ROC-AUC, confusion matrix

Accuracy alone hides class-specific mistakes. We report three standard metrics on the **test** set: **F1** (harmonic mean of precision and recall), **ROC-AUC** (how well the model ranks REAL above FAKE), and a **confusion matrix** (true vs predicted).

ROC-AUC needs **probabilities**, so we collect `sigmoid(logit)` for every test image (1 = REAL is the positive class).

Documentation: [`sklearn.metrics`](https://scikit-learn.org/stable/modules/classes.html#module-sklearn.metrics)

In [ ]:
from sklearn.metrics import f1_score, roc_auc_score, confusion_matrix, ConfusionMatrixDisplay, roc_curve
import numpy as np

model.eval()
y_true, y_prob = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        probs = torch.sigmoid(logits).squeeze(1).cpu()
        y_true.extend(labels.tolist())
        y_prob.extend(probs.tolist())

y_true = np.array(y_true)
y_prob = np.array(y_prob)
y_pred = (y_prob >= 0.5).astype(int)

acc = (y_pred == y_true).mean()
f1 = f1_score(y_true, y_pred)
roc_auc = roc_auc_score(y_true, y_prob)

print(f'Accuracy : {acc:.4f}')
print(f'F1 score : {f1:.4f}')
print(f'ROC-AUC  : {roc_auc:.4f}')


# ---- shared evaluation helpers (used by Parts 9-11) -------------------------
@torch.no_grad()
def predicted_probs(model, loader):
    """Return (true labels, P(REAL)) for every item in a loader."""
    was_training = model.training
    model.eval()
    y_true, y_prob = [], []
    for images, labels in loader:
        logits = model(images.to(device))
        y_prob.extend(torch.sigmoid(logits).squeeze(1).cpu().tolist())
        y_true.extend(labels.tolist())
    model.train(was_training)
    return np.array(y_true), np.array(y_prob)


def metrics_of(y_true, y_prob):
    """Accuracy, F1 and ROC-AUC for one prediction set."""
    y_pred = (y_prob >= 0.5).astype(int)
    return {
        'accuracy': float((y_pred == y_true).mean()),
        'f1': float(f1_score(y_true, y_pred)),
        'roc_auc': float(roc_auc_score(y_true, y_prob)),
    }


In [ ]:
cm = confusion_matrix(y_true, y_pred)   # rows=true, cols=predicted: [[TN,FP],[FN,TP]]
disp = ConfusionMatrixDisplay(cm, display_labels=['FAKE', 'REAL'])
disp.plot(cmap='Blues')
plt.title('Confusion matrix')
plt.show()

fpr, tpr, _ = roc_curve(y_true, y_prob)
plt.figure(figsize=(5, 5))
plt.plot(fpr, tpr, label=f'ROC (AUC = {roc_auc:.3f})')
plt.plot([0, 1], [0, 1], 'k--', label='random')
plt.xlabel('False positive rate')
plt.ylabel('True positive rate')
plt.title('ROC curve')
plt.legend()
plt.show()

---
# Part 10. Robustness to JPEG, blur, and noise

A deployed detector sees images that are compressed, blurred, and noisy. We run the **same trained CNN** on modified copies of the **test** set (never the training set) and measure the accuracy drop.

- **JPEG**: re-encode the raw image at low quality.
- **Gaussian blur** and **Gaussian noise**: applied to the image tensor before normalization.

Requirement: at least 2 of JPEG / blur / noise. We do all three.

In [ ]:
from PIL import Image
import io
import numpy as np
import torch
from sklearn.metrics import f1_score, roc_auc_score
from torch.utils.data import DataLoader
from torchvision import datasets
from torchvision.transforms import v2


class JPEGCompress:
    """Re-encode one PIL image as JPEG at a fixed quality."""

    def __init__(self, quality):
        self.quality = quality

    def __call__(self, pil_img):
        buf = io.BytesIO()
        pil_img.convert('RGB').save(buf, format='JPEG', quality=self.quality)
        buf.seek(0)
        return Image.open(buf).convert('RGB')


def add_gaussian_noise(x, std):
    """Additive Gaussian noise on a [0,1] tensor."""
    return torch.clamp(x + torch.randn_like(x) * std, 0.0, 1.0)


def make_test_loader(pil_mod=None, tensor_mod=None):
    """Rebuild the test loader with one extra degradation."""
    steps = []
    if pil_mod is not None:
        steps.append(pil_mod)
    steps += [v2.ToImage(), v2.ToDtype(torch.float32, scale=True)]
    if tensor_mod is not None:
        steps.append(v2.Lambda(tensor_mod))
    steps.append(v2.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)))
    ds = datasets.ImageFolder(root=data_root / 'test', transform=v2.Compose(steps))
    return DataLoader(ds, batch_size=512, shuffle=False)


def score(loader):
    """Accuracy, F1 and ROC-AUC of the current model on one loader."""
    model.eval()
    y_true, y_prob = [], []
    with torch.no_grad():
        for images, labels in loader:
            p = torch.sigmoid(model(images.to(device))).squeeze(1).cpu()
            y_prob.extend(p.tolist())
            y_true.extend(labels.tolist())
    y_true, y_prob = np.array(y_true), np.array(y_prob)
    y_hat = (y_prob >= 0.5).astype(int)
    return {'accuracy': float((y_hat == y_true).mean()),
            'f1': float(f1_score(y_true, y_hat)),
            'roc_auc': float(roc_auc_score(y_true, y_prob))}


# Clean test set first, then each degradation, all with the same metric set.
robust_metrics = {'clean': score(make_test_loader())}
for name, loader in {
    'JPEG q=20': make_test_loader(pil_mod=JPEGCompress(20)),
    'blur 3x3': make_test_loader(
        tensor_mod=lambda x: v2.functional.gaussian_blur(x, kernel_size=3, sigma=1.0)),
    'noise 0.1': make_test_loader(tensor_mod=lambda x: add_gaussian_noise(x, 0.1)),
}.items():
    robust_metrics[name] = score(loader)

# Kept for the robustness bar chart below.
results = {k: v['accuracy'] for k, v in robust_metrics.items()}

print(f"{'condition':<10}{'accuracy':>10}{'F1':>8}{'ROC-AUC':>10}")
for name, m in robust_metrics.items():
    print(f"{name:<10}{m['accuracy']:>10.4f}{m['f1']:>8.4f}{m['roc_auc']:>10.4f}")


In [ ]:
plt.figure(figsize=(7, 4))
names = list(results)
accs = list(results.values())
bars = plt.bar(names, accs, color='steelblue')
plt.ylim(0, 1)
for b, a in zip(bars, accs):
    plt.text(b.get_x() + b.get_width() / 2, a + 0.01, f'{a:.3f}', ha='center')
plt.ylabel('Test accuracy')
plt.title('Robustness to image perturbations')
plt.show()

---
# Part 11. Fine-tuned pretrained ResNet (transfer learning)

The custom CNN learned from scratch. The **industry-standard baseline** is a ResNet-18 **pretrained on ImageNet**, frozen, with only its final layer retrained for FAKE/REAL — the CS231n / PyTorch transfer-learning pattern.

Two required changes vs the custom CNN:
1. **Resize to 224×224** and use **ImageNet normalization** (the statistics ResNet expects).
2. Replace the 1000-class head with `Linear(512, 1)`.

Citations: He et al. (2016), *Deep residual learning for image recognition*, CVPR; Deng et al. (2009), *ImageNet*, CVPR.

Note: this is CPU-slow (ResNet-18 at 224×224). Run on a GPU / Colab, or lower `SUBSET` in Part 3.

In [ ]:
from torchvision import models

resnet = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
for p in resnet.parameters():
    p.requires_grad = False                       # freeze the pretrained backbone
resnet.fc = nn.Linear(resnet.fc.in_features, 1)   # 512 -> 1 logit
resnet = resnet.to(device)

print('ResNet trainable params:', sum(p.numel() for p in resnet.parameters() if p.requires_grad))

resnet_transform = v2.Compose([
    v2.ToImage(),
    v2.ToDtype(torch.float32, scale=True),
    v2.Resize((224, 224), antialias=True),
    v2.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),   # ImageNet stats
])

r_train_ds = datasets.ImageFolder(root=data_root / 'train', transform=resnet_transform)
r_test_ds  = datasets.ImageFolder(root=data_root / 'test',  transform=resnet_transform)

# same 90k/10k split and 5k slice as Part 3 (same seed -> reproducible)
r_train_set, r_val_set = random_split(r_train_ds, [90000, 10000], generator=torch.Generator().manual_seed(0))
idx = torch.randperm(len(r_train_set), generator=torch.Generator().manual_seed(0))[:SUBSET]
r_train_sub = Subset(r_train_set, idx)

r_train_loader = DataLoader(r_train_sub, batch_size=64, shuffle=True)
r_val_loader   = DataLoader(r_val_set, batch_size=512, shuffle=False)
r_test_loader  = DataLoader(r_test_ds, batch_size=512, shuffle=False)

In [ ]:
optimizer = torch.optim.Adam(resnet.parameters(), lr=1e-3)   # frozen params get no grads, so only fc updates

_, untrained_acc = run_epoch(resnet, r_val_loader)
print(f'ResNet head (untrained) val acc: {untrained_acc:.4f}')

EPOCHS = 3
for epoch in range(1, EPOCHS + 1):
    train_loss, train_acc = run_epoch(resnet, r_train_loader, optimizer)
    val_loss, val_acc = run_epoch(resnet, r_val_loader)
    print(f'Epoch {epoch}/{EPOCHS}  train loss {train_loss:.4f} acc {train_acc:.4f}  val loss {val_loss:.4f} acc {val_acc:.4f}')

resnet_test_loss, resnet_test_acc = run_epoch(resnet, r_test_loader)
print(f'ResNet test accuracy:     {resnet_test_acc:.4f}')
print(f'Custom CNN test accuracy: {test_acc:.4f}')

# Same metric set as the custom CNN, so the comparison in Table I is complete.
r_true, r_prob = predicted_probs(resnet, r_test_loader)
r_metrics = metrics_of(r_true, r_prob)
print(f"ResNet    -> accuracy {r_metrics['accuracy']:.4f}  "
      f"F1 {r_metrics['f1']:.4f}  ROC-AUC {r_metrics['roc_auc']:.4f}")

summary = {'custom CNN': {'accuracy': float(test_acc),
                          'f1': float(f1),
                          'roc_auc': float(roc_auc)},
           'ResNet-18': r_metrics}
print()
print(f"{'model':<12}{'accuracy':>10}{'F1':>8}{'ROC-AUC':>10}")
for name, m in summary.items():
    print(f"{name:<12}{m['accuracy']:>10.4f}{m['f1']:>8.4f}{m['roc_auc']:>10.4f}")


---
# Part 12. Error analysis and summary

The report needs **representative correct and incorrect predictions** and a **limits** discussion. We show eight of each, then summarize results and cite sources.

In [ ]:
def unnormalize(x):
    return x * 0.5 + 0.5   # [-1,1] -> [0,1] for display

model.eval()
correct_examples, wrong_examples = [], []
with torch.no_grad():
    for images, labels in test_loader:
        logits = model(images.to(device))
        probs = torch.sigmoid(logits).squeeze(1).cpu()
        preds = (probs >= 0.5).long()
        for i in range(images.size(0)):
            item = (images[i], labels[i].item(), preds[i].item(), probs[i].item())
            if preds[i] == labels[i] and len(correct_examples) < 8:
                correct_examples.append(item)
            elif preds[i] != labels[i] and len(wrong_examples) < 8:
                wrong_examples.append(item)
        if len(correct_examples) >= 8 and len(wrong_examples) >= 8:
            break

In [ ]:
classes = test_dataset.classes

def show_grid(examples, title):
    fig, axes = plt.subplots(2, 4, figsize=(12, 6))
    for ax, (img, true, pred, conf) in zip(axes.flat, examples):
        ax.imshow(unnormalize(img).permute(1, 2, 0))
        ax.set_title(f'true {classes[true]}  pred {classes[pred]}  P(REAL)={conf:.2f}', fontsize=9)
        ax.axis('off')
    fig.suptitle(title)
    plt.tight_layout()
    plt.show()

show_grid(correct_examples, 'Correct predictions')
show_grid(wrong_examples, 'Wrong predictions')

### Summary (fill in after running)

| Metric | Custom CNN | Pretrained ResNet |
|---|---|---|
| Test accuracy | … | … |
| F1 | … | … |
| ROC-AUC | … | … |

Both columns come from the Part 11 comparison print: accuracy, F1 and ROC-AUC are
computed once for the custom CNN (Part 9) and once for the ResNet (Part 11), on
the same 20,000-image test set.

| Perturbation (custom CNN) | Accuracy | F1 | ROC-AUC |
|---|---|---|---|
| clean | … | … | … |
| JPEG q=20 | … | … | … |
| blur 3×3 | … | … | … |
| noise 0.1 | … | … | … |

The `robust_metrics` dict printed in Part 10 holds these numbers.

### Limits

- Evaluated only on CIFAKE (32×32; CIFAR-10 categories + Stable-Diffusion fakes). Performance on other sources / resolutions is not guaranteed.
- The model learns *statistical* artifacts, not semantics, so unseen generators or heavy post-processing may fool it.
- Robustness tests show accuracy drops under JPEG / blur / noise, so the detector is not reliable on heavily degraded images.
- Training used a 5,000-image subset per epoch, not the full 90,000.

### Citations

- Bird & Lotfi (2024), *CIFAKE*, IEEE Access — dataset.
- He et al. (2016), *Deep residual learning for image recognition*, CVPR — ResNet.
- Deng et al. (2009), *ImageNet*, CVPR — pretrained weights.
- Paszke et al. (2019), *PyTorch*, NeurIPS — framework.
- Pedregosa et al. (2011), *scikit-learn*, JMLR — metrics.
- Wang et al. (2020), *CNN-generated images are surprisingly easy to spot*, CVPR.
- Frank et al. (2020), *Leveraging frequency analysis*, ICML.


---
# Where we go next — together

The data is organized. The remaining stages will be added **one at a time, with you doing the thinking**. For each stage: read the listed videos, answer the guiding questions in chat, then we write and run that part of the notebook together.

| Stage | What we'll build | Watch first | Guiding question |
|---|---|---|---|
| 5 | ✅ Built: the ideas (neuron, layers, ReLU, loss, training loop) | — | — |
| 6 | ✅ Built: baseline MLP on raw pixels | — | — |
| 7 | ✅ Built: custom CNN (Part 7) | #14 | — |
| 8 | Training (the 5-step loop) + loss/accuracy curves | #04, #05 (review) | Why must `zero_grad` come before `backward`? |
| 9 | Metrics: F1, ROC-AUC, confusion matrix | StatQuest "ROC and AUC, Clearly Explained" | Why is accuracy alone not enough? |
| 10 | Robustness: JPEG, blur, noise on test images | — | Which modification do you predict hurts most, and why? |
| 11 | Fine-tuned ResNet comparison | — | What does "pretrained" mean, and what do we change to fit 2 classes? |
| 12 | Final report: results, correct/wrong examples, limits discussion | — | How far would this detector generalize to other image sources? |

**Project complete.** Run Parts 8–11 in order, fill in the Part 12 summary table with your numbers, then export the results into `report.md` in the companion repo.